**1. Import Required Libraries**

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

**2. Load the 5G NIDD Dataset**

In [ ]:
# Load
df = pd.read_csv('/content/drive/MyDrive/5G_NIDD_multiclass_clean.csv', low_memory=False)

print("Original shape:", df.shape)

# Target
y = df['Label']
X = df.drop(columns=['Label', 'Attack Type', 'Attack Tool'], errors='ignore')

# Remove obvious non-learning columns
drop_cols = [
    'SrcMac','DstMac','SrcAddr','DstAddr','StartTime','LastTime',        # drop these columns as they dont have any importance in model(these are IP addresses)
    'SrcOui','DstOui'
]

X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors='ignore')

# Keep only numeric features
X = X.select_dtypes(include=[np.number])   # keeps only numeric features

print("After numeric selection:", X.shape)

Original shape: (1215890, 112)
After numeric selection: (1215890, 86)


**3. Data Cleaning and Handling Missing Values**

In [ ]:
X.replace([np.inf, -np.inf], np.nan, inplace=True)     # remove infinite values
X.fillna(0, inplace=True)                              # handle missing values

**4. Feature Selection using SelectKBest**

In [ ]:
selector = SelectKBest(score_func=f_classif, k=36)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", selected_features.tolist())

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [ 1  7 13 14 19 20 21 22 23 24 25 26 27 34 35 36 37 48 49 50 51 57 58 59
 60 61 62 63 64 65 66 67 68 69 70 71 72 77 78 79 80] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


Selected Features: ['Rank', 'Seq', 'Dur', 'RunTime', 'Mean', 'Sum', 'Min', 'Max', 'sTos', 'dTos', 'sTtl', 'dTtl', 'sHops', 'dHops', 'TotPkts', 'SrcPkts', 'DstPkts', 'TotBytes', 'SrcBytes', 'DstBytes', 'Offset', 'sMeanPktSz', 'dMeanPktSz', 'Loss', 'SrcLoss', 'DstLoss', 'pLoss', 'SrcWin', 'DstWin', 'sVid', 'dVid', 'SrcTCPBase', 'DstTCPBase', 'TcpRtt', 'SynAck', 'AckDat']


**5. Label Encoding(Encode Target Labels)**

In [ ]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

num_classes = len(np.unique(y_encoded))
print("Classes:", num_classes)

Classes: 20


**6. Split Dataset into Training and Testing Sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_encoded,
    test_size=0.2,                    # train data = 80%, test data = 20%
    stratify=y_encoded,
    random_state=42
)

**7. Feature Scaling using QuantileTransformer**

In [ ]:
from sklearn.preprocessing import QuantileTransformer

scaler = QuantileTransformer(
    n_quantiles=min(1000, len(X_train)),   # avoids warning if dataset < 1000
    output_distribution='normal',          # 'normal' usually works better for neural networks
    random_state=42
)

X_train = scaler.fit_transform(X_train)   # FIT ONLY TRAIN
X_test  = scaler.transform(X_test)        # TRANSFORM TEST

**8. Reshape Data for BiGRU Input**

In [ ]:
X_train = X_train.reshape(-1, 36, 1)        # Prepare data in 3D format required by GRU networks
X_test  = X_test.reshape(-1, 36, 1)

**9. Import Deep Learning Layers and Mobile-Net-V1 & BIGRU(Combined) MODEL**

In [ ]:
def MobileNetV1_BiGRU(drop_rate, gru_units, dense_units):

    inp = Input(shape=(36,1))

    # ----- CNN BRANCH -----                    # MOBILE-NET V1 MODEL
    x = Reshape((36,1,1))(inp)

    x = Conv2D(32,(3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(64,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = DepthwiseConv2D((3,3),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(128,(1,1),padding="same")(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    cnn_out = GlobalAveragePooling2D()(x)

    # ----- GRU BRANCH -----                                 # BIGRU MODEL
    r = Bidirectional(GRU(gru_units, return_sequences=True))(inp)
    r = Bidirectional(GRU(gru_units))(r)

    # ----- FUSION -----                                    # PROJECTION(Combines both models)
    merged = Concatenate()([cnn_out, r])

    merged = Dense(dense_units, activation="relu")(merged)
    merged = Dense(dense_units//2, activation="relu")(merged)
    merged = Dropout(drop_rate)(merged)

    out = Dense(num_classes, activation="softmax")(merged)

    model = Model(inp, out)
    return model


**10. Install keras tuner**

In [ ]:
!pip install keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.5 MB/s eta 0:00:00


In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 10.9 MB/s eta 0:00:00


**11. Setup TensorFlow & Focal Loss**

In [ ]:
import tensorflow as tf

def focal_loss(gamma, alpha):

    def loss(y_true, y_pred):

        y_true = tf.cast(y_true, tf.int32)
        y_true = tf.one_hot(y_true, depth=num_classes)

        ce = tf.keras.losses.categorical_crossentropy(y_true, y_pred)

        pt = tf.exp(-ce)
        focal = alpha * tf.pow(1 - pt, gamma) * ce

        return tf.reduce_mean(focal)

    return loss


**12. Handle Class Imbalance using Class Weights**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, class_weights))
print(class_weights)


{np.int64(0): np.float64(0.64811172410117), np.int64(1): np.float64(0.6491671115856914), np.int64(2): np.float64(3.325965944060726), np.int64(3): np.float64(4.20650406504065), np.int64(4): np.float64(37.819284603421465), np.int64(5): np.float64(44.74296228150874), np.int64(6): np.float64(1.3619983757596124), np.int64(7): np.float64(4.309374446216552), np.int64(8): np.float64(5.309563318777292), np.int64(9): np.float64(5.274438781043271), np.int64(10): np.float64(1.9601644365629534), np.int64(11): np.float64(4.803516049382716), np.int64(12): np.float64(5.220652640618291), np.int64(13): np.float64(5.217292426517915), np.int64(14): np.float64(1.5948189926547744), np.int64(15): np.float64(1.9197000197355436), np.int64(16): np.float64(0.12998123867505493), np.int64(17): np.float64(0.21242149215139894), np.int64(18): np.float64(6.053721682847897), np.int64(19): np.float64(5.8995147986414365)}


**Dividing data further**

100% DATA = 80% TRAIN | 20% TEST   (test=used for final evaluation)

TRAIN = 80% CLIENTS | 20% SERVER   (server=used for each round evaluation)

CLIENT = 70% FL DATA | 30% PRETRAIN  (fl=hypertuning and divided b/w clients)

In [ ]:
from sklearn.model_selection import train_test_split

# Train-Test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)

# Server (20%) and Client (80%)
X_server, X_clients, y_server, y_clients = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.8,
    stratify=y_train_full,
    random_state=42
)

# # Split client data → FL + (optional small pretrain if needed later)
# X_fl, _, y_fl, _ = train_test_split(
#     X_clients,
#     y_clients,
#     test_size=0.3,
#     stratify=y_clients,
#     random_state=42
# )

**13. Optuna Training of hyperparameters**

In [ ]:
import optuna
import tensorflow as tf
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

X_tune = X_clients
y_tune = y_clients

BEST_MODEL_PATH = "best_optuna_model.weights.h5"
best_score = 0

def objective(trial):
    global best_score
    print(f"\nStarting Trial {trial.number}")

    tf.keras.backend.clear_session()

    dropout = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
    gru_units = trial.suggest_int("gru_units", 64, 192, step=64)
    dense_units = trial.suggest_int("dense_units", 256, 384, step=64)
    gamma = trial.suggest_float("gamma", 1.0, 3.0, step=1)
    alpha = trial.suggest_float("alpha", 0.2, 0.4, step=0.1)
    lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)

    model = MobileNetV1_BiGRU(
        drop_rate=dropout,
        gru_units=gru_units,
        dense_units=dense_units
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=gamma, alpha=alpha),
        metrics=["accuracy"]
    )

    history = model.fit(
        X_tune,
        y_tune,
        validation_split=0.2,
        epochs=15,
        batch_size=256,
        callbacks=[
            EarlyStopping(patience=3, restore_best_weights=True),
            ReduceLROnPlateau(patience=2)
        ],
        verbose=1
    )

    val_acc = max(history.history["val_accuracy"])

    if val_acc > best_score:
        best_score = val_acc
        model.save_weights(BEST_MODEL_PATH)

    return val_acc


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=6)

best_params = study.best_params

[I 2026-04-17 08:05:31,775] A new study created in memory with name: no-name-cb0be05d-b65f-437c-ac8e-1dcc14375c9d



Starting Trial 0
Epoch 1/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 64s 24ms/step - accuracy: 0.6535 - loss: 0.0763 - val_accuracy: 0.7518 - val_loss: 0.0434 - learning_rate: 1.5942e-04
Epoch 2/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 51s 26ms/step - accuracy: 0.7772 - loss: 0.0377 - val_accuracy: 0.8413 - val_loss: 0.0280 - learning_rate: 1.5942e-04
Epoch 3/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 48s 24ms/step - accuracy: 0.8400 - loss: 0.0263 - val_accuracy: 0.8680 - val_loss: 0.0215 - learning_rate: 1.5942e-04
Epoch 4/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 49s 25ms/step - accuracy: 0.8626 - loss: 0.0214 - val_accuracy: 0.8671 - val_loss: 0.0188 - learning_rate: 1.5942e-04
Epoch 5/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 25ms/step - accuracy: 0.8727 - loss: 0.0188 - val_accuracy: 0.8854 - val_loss: 0.0166 - learning_rate: 1.5942e-04
Epoch 6/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 48s 24ms/step - accuracy: 0.8802 - loss: 0.0170 - val_accuracy: 0.8835 - val_loss: 0.0156 - learning_rate: 1.5942e-04
Epoch 7/15
1946/1946 ━

[I 2026-04-17 08:19:07,960] Trial 0 finished with value: 0.9280929565429688 and parameters: {'dropout': 0.2, 'gru_units': 64, 'dense_units': 384, 'gamma': 3.0, 'alpha': 0.2, 'lr': 0.00015942365564119244}. Best is trial 0 with value: 0.9280929565429688.



Starting Trial 1
Epoch 1/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 67s 31ms/step - accuracy: 0.6954 - loss: 0.1618 - val_accuracy: 0.8042 - val_loss: 0.0961 - learning_rate: 1.8417e-04
Epoch 2/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 59s 30ms/step - accuracy: 0.8274 - loss: 0.0816 - val_accuracy: 0.8629 - val_loss: 0.0629 - learning_rate: 1.8417e-04
Epoch 3/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 60s 31ms/step - accuracy: 0.8663 - loss: 0.0603 - val_accuracy: 0.8825 - val_loss: 0.0518 - learning_rate: 1.8417e-04
Epoch 4/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 59s 31ms/step - accuracy: 0.8809 - loss: 0.0520 - val_accuracy: 0.8793 - val_loss: 0.0477 - learning_rate: 1.8417e-04
Epoch 5/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 31ms/step - accuracy: 0.8904 - loss: 0.0468 - val_accuracy: 0.8984 - val_loss: 0.0419 - learning_rate: 1.8417e-04
Epoch 6/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 59s 30ms/step - accuracy: 0.8957 - loss: 0.0436 - val_accuracy: 0.9093 - val_loss: 0.0383 - learning_rate: 1.8417e-04
Epoch 7/15
1946/1946 ━

[I 2026-04-17 08:34:51,709] Trial 1 finished with value: 0.9284704327583313 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 384, 'gamma': 1.0, 'alpha': 0.30000000000000004, 'lr': 0.0001841668304638462}. Best is trial 1 with value: 0.9284704327583313.


...

[I 2026-04-17 09:05:34,102] Trial 3 finished with value: 0.9206476807594299 and parameters: {'dropout': 0.30000000000000004, 'gru_units': 128, 'dense_units': 256, 'gamma': 3.0, 'alpha': 0.2, 'lr': 0.00029714534203258917}. Best is trial 2 with value: 0.9467263221740723.



Starting Trial 4
Epoch 1/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 54s 24ms/step - accuracy: 0.7227 - loss: 0.0957 - val_accuracy: 0.8059 - val_loss: 0.0549 - learning_rate: 3.7262e-04
Epoch 2/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 45s 23ms/step - accuracy: 0.8394 - loss: 0.0476 - val_accuracy: 0.8555 - val_loss: 0.0398 - learning_rate: 3.7262e-04
Epoch 3/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - accuracy: 0.8672 - loss: 0.0381 - val_accuracy: 0.8787 - val_loss: 0.0333 - learning_rate: 3.7262e-04
Epoch 4/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - accuracy: 0.8797 - loss: 0.0340 - val_accuracy: 0.8947 - val_loss: 0.0300 - learning_rate: 3.7262e-04
Epoch 5/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 45s 23ms/step - accuracy: 0.8881 - loss: 0.0315 - val_accuracy: 0.8998 - val_loss: 0.0281 - learning_rate: 3.7262e-04
Epoch 6/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 46s 23ms/step - accuracy: 0.8937 - loss: 0.0295 - val_accuracy: 0.9071 - val_loss: 0.0259 - learning_rate: 3.7262e-04
Epoch 7/15
1946/1946 ━

[I 2026-04-17 09:17:44,545] Trial 4 finished with value: 0.9263420701026917 and parameters: {'dropout': 0.4, 'gru_units': 64, 'dense_units': 384, 'gamma': 1.0, 'alpha': 0.2, 'lr': 0.00037261925496471875}. Best is trial 2 with value: 0.9467263221740723.



Starting Trial 5
Epoch 1/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 65s 30ms/step - accuracy: 0.7666 - loss: 0.1177 - val_accuracy: 0.8566 - val_loss: 0.0589 - learning_rate: 6.5151e-04
Epoch 2/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8674 - loss: 0.0572 - val_accuracy: 0.8850 - val_loss: 0.0465 - learning_rate: 6.5151e-04
Epoch 3/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 82s 30ms/step - accuracy: 0.8799 - loss: 0.0498 - val_accuracy: 0.8989 - val_loss: 0.0431 - learning_rate: 6.5151e-04
Epoch 4/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8876 - loss: 0.0455 - val_accuracy: 0.8998 - val_loss: 0.0407 - learning_rate: 6.5151e-04
Epoch 5/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8940 - loss: 0.0424 - val_accuracy: 0.8994 - val_loss: 0.0391 - learning_rate: 6.5151e-04
Epoch 6/15
1946/1946 ━━━━━━━━━━━━━━━━━━━━ 58s 30ms/step - accuracy: 0.8994 - loss: 0.0401 - val_accuracy: 0.9137 - val_loss: 0.0340 - learning_rate: 6.5151e-04
Epoch 7/15
1946/1946 ━

[I 2026-04-17 09:32:45,284] Trial 5 finished with value: 0.9458428621292114 and parameters: {'dropout': 0.4, 'gru_units': 128, 'dense_units': 256, 'gamma': 1.0, 'alpha': 0.30000000000000004, 'lr': 0.0006515068303484095}. Best is trial 2 with value: 0.9467263221740723.


**14. Compiled model**

In [ ]:
def get_compiled_model():
    model = MobileNetV1_BiGRU(
        drop_rate=best_params["dropout"],
        gru_units=best_params["gru_units"],
        dense_units=best_params["dense_units"]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=best_params["lr"]),
        loss=focal_loss(
            gamma=best_params["gamma"],
            alpha=best_params["alpha"]
        ),
        metrics=["accuracy"]
    )

    return model

In [ ]:
global_model = get_compiled_model()
global_model.load_weights(BEST_MODEL_PATH)

**15. Create Federated Clients**

In [ ]:
from sklearn.model_selection import StratifiedKFold

NUM_CLIENTS = 7

def create_clients(X, y, num_clients):
    skf = StratifiedKFold(n_splits=num_clients, shuffle=True, random_state=42)
    client_data = {}

    for i, (_, idx) in enumerate(skf.split(X, y)):
        client_data[i] = (X[idx], y[idx])

    return client_data


client_data = create_clients(X_clients, y_clients, NUM_CLIENTS)

**16. Client Local Training Function**

In [ ]:
def client_update(global_weights, dataset, local_epochs=2):

    local_model = get_compiled_model()
    local_model.set_weights(global_weights)

    X, y = dataset

    local_model.fit(
        X,
        y,
        epochs=local_epochs,
        batch_size=256,
        verbose=0
    )

    return local_model.get_weights(), len(X)

**17. Federated Averaging (FedAvg)**

In [ ]:
def aggregate_weights(client_weights, client_sizes):

    total_samples = sum(client_sizes)
    avg_weights = []

    for weights_list_tuple in zip(*client_weights):
        weighted_sum = np.zeros_like(weights_list_tuple[0])

        for w, size in zip(weights_list_tuple, client_sizes):
            weighted_sum += w * (size / total_samples)

        avg_weights.append(weighted_sum)

    return avg_weights

**18. Federated Training Loop (Server)**

In [ ]:
import random

ROUNDS = 13
CLIENTS_PER_ROUND = 7

for round_num in range(ROUNDS):

    print(f"\nFL Round {round_num}")

    global_weights = global_model.get_weights()

    client_weights = []
    client_sizes = []

    selected_clients = random.sample(
        list(client_data.keys()),
        CLIENTS_PER_ROUND
    )

    for client in selected_clients:
        weights, size = client_update(
            global_weights,
            client_data[client]
        )
        client_weights.append(weights)
        client_sizes.append(size)

    new_global_weights = aggregate_weights(client_weights, client_sizes)

    global_model.set_weights(new_global_weights)

    # Server evaluation (Approach 2)
    loss, acc = global_model.evaluate(X_server, y_server, verbose=0)
    print(f"Server Accuracy: {acc:.4f}")


FL Round 0
Server Accuracy: 0.9368

FL Round 1
Server Accuracy: 0.9389

FL Round 2
Server Accuracy: 0.9430

FL Round 3
Server Accuracy: 0.9382

FL Round 4
Server Accuracy: 0.9432

FL Round 5
Server Accuracy: 0.9412

FL Round 6
Server Accuracy: 0.9462

FL Round 7
Server Accuracy: 0.9399

FL Round 8
Server Accuracy: 0.9472

FL Round 9
Server Accuracy: 0.9430

FL Round 10
Server Accuracy: 0.9452

FL Round 11
Server Accuracy: 0.9589

FL Round 12
Server Accuracy: 0.9491


**19. Final Accuracy**

In [ ]:
loss, acc = global_model.evaluate(X_test, y_test, verbose=1)
print("\nFinal Test Accuracy:", acc)

6080/6080 ━━━━━━━━━━━━━━━━━━━━ 45s 7ms/step - accuracy: 0.9495 - loss: 0.0209

Final Test Accuracy: 0.9494713544845581
